# code to import ITS_LIVE velocities

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.dates
import itslive
import geopandas as gpd
from shapely.geometry import Point
from tqdm.auto import tqdm
import os

/Users/lindsaysummers/micromamba/envs/itslive_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## modify inputs

In [2]:
#date_dt preference (integer)
date_dt_min = 20
date_dt_max = 40

#start date and end date for velocities (string)
start = '2015-01-01'
end = '2025-01-01'

#path to list of glaciers
glacier_list_path = '/Users/lindsaysummers/Documents/Research/Alaska_seasonality/glacier_selection/Alaska_seasonality_manual_selection.csv'
#glacier_list_path = '/Users/lindsaysummers/Documents/Research/Alaska_seasonality/glacier_selection/Canada_seasonality_manual_selection.csv'

#path to rgi shapefile
rgi_shapefile_path = '/Users/lindsaysummers/Documents/Research/RGI/RGI2000-v7.0-L-01_alaska/RGI2000-v7.0-L-01_alaska.shp'
#rgi_shapefile_path = '/Users/lindsaysummers/Documents/Research/RGI/RGI2000-v7.0-L-02_western_canada_usa/RGI2000-v7.0-L-02_western_canada_usa.shp'

#space between points along each centerline (in m) (integer)
point_spacing = 1000

#projection for region (string)
projection = "EPSG:3338"

#distance between desired point and surrounding points (m) (integer)
footprint_spacing = 120

#path to output storage
output_folder = '/Users/lindsaysummers/Documents/Research/Alaska_seasonality/itslive_velocities/'
os.makedirs(output_folder, exist_ok=True)

In [3]:
#import list of rgi ids of wanted glaciers

#clip rgi df to only include desired glaciers

#pull equally spaced points for each centerline in lat lon coordinates

#get itslive velocities for each centerline point

#store in a folder with one csv for each rgi string

#weighted averaging for each velocity time series based on date_dt

#bimonthly median for each glacier (do this later, for now just monthly resolution)

#generate spatiotemporal heatmap AND/OR bimonthly median speed profiles for each glacier

## import list of glaciers & RGI info then merge

In [4]:
#read glacier list
glacier_list = pd.read_csv(glacier_list_path)

#remove empty unnamed columns
glacier_list = glacier_list.loc[
    :, ~glacier_list.columns.str.startswith("Unnamed")
]

#read RGI centerlines
centerlines = gpd.read_file(rgi_shapefile_path)

#merge
df = centerlines.merge(
    glacier_list,
    left_on="rgi_g_id",
    right_on="rgi_id",
    how="inner"
)

#keep one consistent RGI ID column
#df = df.drop(columns=["RGI_id"])
df = df.rename(columns={"rgi_g_id": "rgi_id"})

print(df.columns.tolist())
print(f"{len(df)} centerlines matched")

['rgi_id_x', 'rgi_id', 'segment_id', 'is_main', 'outflow_id', 'strahler_n', 'length_m', 'geometry', 'rgi_id_y', 'cen_lat', 'cen_lon', 'area_km2', 'aspect_deg', 'slope_deg', 'z_min', 'z_max', 'burgess_classification', 'median_annual_precip_cumsum', 'median_annual_temp', 'median_annual_temp_range']
474 centerlines matched


## extract equally spaced points along each centerline

In [5]:
#ensure same projection
centerlines_proj = df.to_crs(projection)

In [6]:
def extract_centerline_points(line, spacing_m=point_spacing):

    # total length of line (m)
    length = line.length

    # distances along line
    distances = np.arange(0, length, spacing_m)

    # interpolate points
    points = [line.interpolate(d) for d in distances]

    return points, distances

In [7]:
#apply
points_list = []

for idx, row in centerlines_proj.iterrows():

    line = row.geometry
    rgi_id = row["rgi_id"]
    segment_id = row["segment_id"]

    pts, distances = extract_centerline_points(line, spacing_m=point_spacing)
 
    for i, (pt, distance) in enumerate(zip(pts, distances)):
        points_list.append({
            "rgi_id": rgi_id,
            "segment_id": segment_id,
            "point_id": i,
            "distance_from_head_m": distance,
            "geometry": pt
        })

points_gdf = gpd.GeoDataFrame(
    points_list,
    geometry="geometry",
    crs=centerlines_proj.crs
)

In [8]:
#convert to lat lon coordinates
points = points_gdf.to_crs("EPSG:4326")

points["lon"] = points.geometry.x
points["lat"] = points.geometry.y

print(points.head())

                    rgi_id  segment_id  point_id  distance_from_head_m  \
0  RGI2000-v7.0-G-01-01595           0         0                   0.0   
1  RGI2000-v7.0-G-01-01595           0         1                1000.0   
2  RGI2000-v7.0-G-01-01595           0         2                2000.0   
3  RGI2000-v7.0-G-01-01595           0         3                3000.0   
4  RGI2000-v7.0-G-01-01595           0         4                4000.0   

                      geometry         lon        lat  
0  POINT (-152.53167 61.10246) -152.531670  61.102464  
1  POINT (-152.53094 61.09407) -152.530943  61.094074  
2  POINT (-152.53081 61.08535) -152.530807  61.085349  
3  POINT (-152.52455 61.07723) -152.524551  61.077231  
4  POINT (-152.51501 61.06969) -152.515010  61.069692  


In [9]:
# (OPTIONAL) export lat lon coordinates for each point
output_path = "/Users/lindsaysummers/Documents/Research/Alaska_seasonality/centerline_points/"
os.makedirs(output_path, exist_ok=True)

# convert points to WGS84 for lon/lat
points_wgs84 = points_gdf.to_crs("EPSG:4326")

# add longitude and latitude
points_wgs84["lon"] = points_wgs84.geometry.x
points_wgs84["lat"] = points_wgs84.geometry.y

# export one CSV per RGI
for rgi_id, glacier_df in points_wgs84.groupby("rgi_id"):

    glacier_df = glacier_df.drop(columns="geometry")

    output_file = os.path.join(
        output_path,
        f"{rgi_id}_points.csv"
    )

    glacier_df.to_csv(output_file, index=False)

    print(f"Saved {output_file}")

Saved /Users/lindsaysummers/Documents/Research/Alaska_seasonality/centerline_points/RGI2000-v7.0-G-01-01595_points.csv
Saved /Users/lindsaysummers/Documents/Research/Alaska_seasonality/centerline_points/RGI2000-v7.0-G-01-01751_points.csv
Saved /Users/lindsaysummers/Documents/Research/Alaska_seasonality/centerline_points/RGI2000-v7.0-G-01-02509_points.csv
Saved /Users/lindsaysummers/Documents/Research/Alaska_seasonality/centerline_points/RGI2000-v7.0-G-01-02613_points.csv
Saved /Users/lindsaysummers/Documents/Research/Alaska_seasonality/centerline_points/RGI2000-v7.0-G-01-03986_points.csv
Saved /Users/lindsaysummers/Documents/Research/Alaska_seasonality/centerline_points/RGI2000-v7.0-G-01-04706_points.csv
Saved /Users/lindsaysummers/Documents/Research/Alaska_seasonality/centerline_points/RGI2000-v7.0-G-01-04948_points.csv
Saved /Users/lindsaysummers/Documents/Research/Alaska_seasonality/centerline_points/RGI2000-v7.0-G-01-05702_points.csv
Saved /Users/lindsaysummers/Documents/Research/A

## extract itslive velocities at each point

In [10]:
def get_itslive(points, footprint_spacing=120):

    #dictionary to store velocity DataFrames
    velocity_dataframes = {}

    #max segment_id for each glacier
    max_segment_by_glacier = (points.groupby("rgi_id")["segment_id"].max())

    #max point_id for each glacier/segment
    max_point_by_segment = (points.groupby(["rgi_id", "segment_id"])["point_id"].max())

    #3x3 footprint
    offsets = [-footprint_spacing,0,footprint_spacing]

    #loop through every centerline point
    for i, row in tqdm(
        points.iterrows(),
        total=len(points),
        desc="Getting ITS_LIVE velocities"):

        #point metadata
        rgi_id = row["rgi_id"]
        segment_id = row["segment_id"]
        pid = row["point_id"]

        #skip first point of every segment
        if pid == 0:
            continue

        #main segment = maximum segment_id for each rgi/glacier
        max_segment_id = max_segment_by_glacier[rgi_id]

        #last point of current segment
        max_point_id = max_point_by_segment[(rgi_id, segment_id)]

        #skip last point of non-main segments/tributaries
        if segment_id != max_segment_id and pid == max_point_id:
            continue
            
        try:
            #convert point from projected coordinates to WGS84
            point_projected = row.geometry

            point_wgs84 = gpd.GeoSeries([point_projected], crs=points.crs).to_crs("EPSG:4326").iloc[0]

            x_proj = point_projected.x
            y_proj = point_projected.y

            #create 3x3 footprint
            footprint_pts_projected = []

            for dx in offsets:
                for dy in offsets:
                    footprint_pts_projected.append((x_proj + dx, y_proj + dy))

            #convert footprint points to lat lon
            footprint_gdf = gpd.GeoDataFrame(geometry=[Point(x, y) for x, y in footprint_pts_projected],crs=points.crs).to_crs("EPSG:4326")

            footprint_pts_wgs84 = [(geom.x, geom.y) for geom in footprint_gdf.geometry]

            #get itslive time series for all 9 footprint points
            timeseries_datasets = (itslive.velocity_cubes.get_time_series(footprint_pts_wgs84))

            #if no data returned skip this point
            if len(timeseries_datasets) == 0:
                continue

            #extract data from each footprint point
            footprint_dfs = []

            for ds in timeseries_datasets:

                ts = ds["time_series"]

                velocity_data = np.atleast_1d(ts["v"])

                v_error_data = np.atleast_1d(ts["v_error"])

                mid_date_data = np.atleast_1d(ts["mid_date"])

                date_dt_data = np.asarray(ts["date_dt"])

                #convert dates
                mid_date = pd.to_datetime(mid_date_data)
                
                #convert date_dt to days
                date_dt_days = (date_dt_data.astype("timedelta64[D]").astype(float))

                #create df for this footprint point
                temp_df = pd.DataFrame({
                    "mid_date": mid_date,
                    "velocity": velocity_data,
                    "v_error": v_error_data,
                    "date_dt_days": date_dt_days})

                footprint_dfs.append(temp_df)

            # combine all 9 footprint time series
            if len(footprint_dfs) == 0:
                continue

            combined_df = pd.concat(footprint_dfs,ignore_index=True)
            
            #remove rows with no velocity
            combined_df = combined_df.dropna(subset=["velocity"])

            if len(combined_df) == 0:
                continue

            #average 3x3 footprint by date
            #averaged_df = (combined_df.groupby("mid_date", as_index=False).agg(velocity_mean=("velocity", "mean"), v_error_mean=("v_error","mean"),date_dt_days=("date_dt_days","first"),n_footprint_points=("velocity","count")))

            #average 3x3 footprint by date
            def propagate_v_error(group):

                #remove NaN uncertainties
                errors = group["v_error"].dropna().to_numpy()

                n = len(errors)

                if n == 0:
                    return np.nan

                #propagated uncertainty of the mean
                return np.sqrt(np.sum(errors**2)) / n
                #return np.sqrt(np.sum(errors**2))

            averaged_df = (combined_df.groupby("mid_date", as_index=False).agg(velocity_mean=("velocity", "mean"),date_dt_days=("date_dt_days", "first"),n_footprint_points=("velocity", "count")))

            # calculate propagated v_error separately
            propagated_errors = (combined_df.groupby("mid_date").apply(propagate_v_error).reset_index(name="v_error"))

            # merge propagated uncertainty back in
            averaged_df = averaged_df.merge(propagated_errors,on="mid_date",how="left")
            
            #require at least one footprint point
            averaged_df = averaged_df[averaged_df["n_footprint_points"] > 0].copy()

            #filter observation duration
            averaged_df = averaged_df[(averaged_df["date_dt_days"] >= date_dt_min) & (averaged_df["date_dt_days"] <= date_dt_max)].copy()

            #filter analysis date range
            averaged_df = averaged_df[(averaged_df["mid_date"] >= pd.Timestamp(start)) & (averaged_df["mid_date"] <= pd.Timestamp(end))].copy()

            #if all observations were removed by filters then skip
            if len(averaged_df) == 0:
                continue
                
            #point metadata
            averaged_df["rgi_id"] = rgi_id

            averaged_df["segment_id"] = segment_id

            averaged_df["point_id"] = pid

            averaged_df["distance_from_head_m"] = (row["distance_from_head_m"])
            
            #store in dictionary
            velocity_dataframes[f"{rgi_id}_seg{segment_id}_pt{pid}"] = averaged_df

        #catch errors so one bad point doesn't stop everything :)
        except Exception as e:

            print(f"Skipping {rgi_id} "f"seg{segment_id} "f"pt{pid}: {e}")

    #print summary
    print(f"\nCompleted {len(velocity_dataframes)} "f"of {len(points)} points "f"({len(velocity_dataframes) / len(points) * 100:.1f}%)")

    return velocity_dataframes

In [11]:
#get velocities

#store in dictionary
#keys = 
velocity_dataframes = get_itslive(points_gdf,footprint_spacing=120)

#check output for one glacier
glacier_df = pd.concat([velocity_dataframes[key] for key in velocity_dataframes if key.startswith(rgi_id)], ignore_index=True)

display(glacier_df.head())

Getting ITS_LIVE velocities:   5%|▌           | 34/715 [01:24<26:54,  2.37s/it]

Getting ITS_LIVE velocities:  13%|█▌          | 95/715 [03:46<15:06,  1.46s/it]

Getting ITS_LIVE velocities:  83%|█████████  | 590/715 [36:27<07:44,  3.72s/it]

Getting ITS_LIVE velocities:  83%|█████████  | 593/715 [36:42<09:08,  4.50s/it]

Getting ITS_LIVE velocities:  83%|█████████▏ | 594/715 [36:47<09:21,  4.64s/it]

Getting ITS_LIVE velocities:  83%|█████████▏ | 595/715 [36:52<09:47,  4.89s/it]

Getting ITS_LIVE velocities: 100%|███████████| 715/715 [46:19<00:00,  3.89s/it]


Completed 486 of 715 points (68.0%)


,mid_date,velocity_mean,date_dt_days,n_footprint_points,v_error,rgi_id,segment_id,point_id,distance_from_head_m
0,2015-02-16 18:47:03.495719936,60.000000,32.0,2,66.822525,RGI2000-v7.0-G-02-10336,0,1,1000.0
1,2015-06-28 18:47:47.255082752,32.500000,24.0,2,43.139309,RGI2000-v7.0-G-02-10336,0,1,1000.0
2,2015-07-27 18:41:42.188814080,25.714285,32.0,7,9.344866,RGI2000-v7.0-G-02-10336,0,1,1000.0
3,2015-08-19 18:48:00.301174016,177.000000,32.0,2,26.162951,RGI2000-v7.0-G-02-10336,0,1,1000.0
4,2015-09-16 18:48:10.862579968,68.000000,24.0,1,54.000000,RGI2000-v7.0-G-02-10336,0,1,1000.0


# save as csv for each RGI ID

In [12]:
#store in one df for each glacier/rgi string
#maybe separate csvs for each segment?

#combine all velocity dfs
velocity_df = pd.concat(velocity_dataframes.values(),ignore_index=True)

#save one CSV for each RGI/glacier
for rgi_id, glacier_df in velocity_df.groupby("rgi_id"):

    glacier_df = glacier_df.sort_values(["segment_id", "point_id", "mid_date"])

    rgi_folder = os.path.join(output_folder, rgi_id)
    os.makedirs(rgi_folder, exist_ok=True)
    
    output_path = os.path.join(rgi_folder,f"{rgi_id}_velocity.csv")

    glacier_df.to_csv(output_path, index=False)

    print(f"Saved {rgi_id}: {len(glacier_df)} rows")

Saved RGI2000-v7.0-G-02-04475: 6225 rows
Saved RGI2000-v7.0-G-02-04479: 7387 rows
Saved RGI2000-v7.0-G-02-04482: 5060 rows
Saved RGI2000-v7.0-G-02-04567: 12791 rows
Saved RGI2000-v7.0-G-02-04571: 1316 rows
Saved RGI2000-v7.0-G-02-04863: 7000 rows
Saved RGI2000-v7.0-G-02-04928: 8487 rows
Saved RGI2000-v7.0-G-02-05699: 26446 rows
Saved RGI2000-v7.0-G-02-05777: 39143 rows
Saved RGI2000-v7.0-G-02-06816: 8689 rows
Saved RGI2000-v7.0-G-02-06848: 45144 rows
Saved RGI2000-v7.0-G-02-07266: 35574 rows
Saved RGI2000-v7.0-G-02-08338: 40315 rows
Saved RGI2000-v7.0-G-02-08538: 122778 rows
Saved RGI2000-v7.0-G-02-10031: 2843 rows
Saved RGI2000-v7.0-G-02-10336: 15324 rows


In [16]:
#save as csv 
#velocity_df.to_csv('/Users/lindsaysummers/Documents/Research/Alaska_seasonality/alaska_glacier_velocities.csv')